# CEL-Seq2 to 10x Chromium V2
Run with the `scPath` kernel and a CUDA GPU. Set PYTHONHASHSEED=42 before launching Jupyter for stable set-based gene ordering.
Place the prepared eight input files under data/RNA/cel_seq2 and the human Reactome/KEGG GMTs under data/pathway/human. See README.md for provenance and outputs.

In [1]:
import os, sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scpath/models.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.is_available(), 'CUDA is required'
device = torch.device(os.environ.get('SCPATH_DEVICE', 'cuda:0'))
print('GPU:', torch.cuda.get_device_name(device))


GPU: NVIDIA GeForce RTX 4090


## Prepare the paired data
The companion processor reads the original exports, applies the source QC, builds Reactome/KEGG graphs, and selects reference HVGs for both splits. No historical feature manifest or checkpoint is substituted.

In [2]:
from examples.cel_seq2.process_data import prepare
reference, query, pathways, output = prepare(PROJECT_ROOT)
print('Reference:', reference.shape, 'Query:', query.shape)


<active-conda-env>/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
<active-conda-env>/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
<active-conda-env>/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:283: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


Original cell count: 463
Cell count after filtering: 463
Gene count after filtering: 12071
Cell count after filtering: 3098
Gene count after filtering: 17519
Number of shared genes: 12052

Processing pathway: Reactome
Gene-set size: 620 genes


./scpath/utils/pathway.py:49: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adata_train_rna.concatenate(adata_test_rna, batch_key='dataset')


Running aucell on mat with 3561 samples and 12052 targets for 1472 sources.


  0%|          | 0/3561 [00:00<?, ?it/s]

KNN graph saved to ./output/cel_seq2/output/train_knn_graph_Reactome.npz.
KNN graph saved to ./output/cel_seq2/output/test_knn_graph_Reactome.npz.

Processing pathway: KEGG
Gene-set size: 69 genes


./scpath/utils/pathway.py:49: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = adata_train_rna.concatenate(adata_test_rna, batch_key='dataset')


Running aucell on mat with 3561 samples and 12052 targets for 357 sources.


  0%|          | 0/3561 [00:00<?, ?it/s]

KNN graph saved to ./output/cel_seq2/output/train_knn_graph_KEGG.npz.
KNN graph saved to ./output/cel_seq2/output/test_knn_graph_KEGG.npz.
KNN graph processing completed for all pathways.
Normalizing data...
Applying log1p transformation...
Normalizing data...
Applying log1p transformation...
Reference: (463, 4000) Query: (3098, 4000)


<active-conda-env>/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:207: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
<active-conda-env>/lib/python3.9/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(
<active-conda-env>/lib/python3.9/site-packages/scanpy/preprocessing/_normalization.py:207: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


## Train and distill
The adaptive batch formula and the fixed 500 steps follow platform.ipynb. high=True retains the original private-class logic; the student then refines rejected assignments.

In [ ]:
from scpath.config import Config
from scpath.train import train
total = reference.n_obs + query.n_obs
smallest = min(reference.n_obs, query.n_obs)
gnn = 'gat' if total < 10000 else 'sage'
batch = smallest // 2 if smallest < 1000 else 512 if smallest < 3000 else 1024
batch = min(batch, smallest)
config = Config(str(output) + '/', reference.n_vars, device, True, batch_size=batch, gnn_type=gnn, max_steps=500, K=len(query.obs['cell_type'].unique()), rate=0.05)
print('GNN:', gnn, 'Batch:', batch, 'Steps:', config.max_steps)
train(config, reference, query, pathways)


2026-11-01 04:09:39.648348: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-01 04:09:39.671028: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[easydl] tensorflow not available!
GNN: gat Batch: 231 Steps: 500
Shared classes: 9
Reference-only classes: 0
Query-only classes: 0
['B cell', 'CD14+ monocyte', 'CD16+ monocyte', 'CD4+ T cell', 'Cytotoxic T cell', 'Dendritic cell', 'Megakaryocyte', 'Natural killer cell', 'Plasmacytoid dendritic cell']
463 463
3098 3098
463 463
3098 3098
                     cell_type  label
0                       B cell      0
1               CD14+ monocyte      1
2               CD16+ monocyte      2
3                  CD4+ T cell      3
4             Cytotoxic T cell      4
5               Dendritic cell      5
6                Megakaryocyte      6
7          Natural killer cell      7
8  Plasmacytoid dendritic cell      8
{'source_classes': [0, 1, 2, 3, 4, 5, 6, 7, 8], 'target_private_classes': [], 'common_classes': [0, 1, 2, 3, 4, 5, 6, 7, 8], 'all_classes': [0, 1, 2, 3, 4, 5, 6, 7, 8]}
2


Evaluating on target domain: 100%|██████████| 14/14 [00:00<00:00, 16.94it/s]

Evaluating on source domain: 100%|██████████| 3/3 [00:00<00:00, 23.43it/s]



[Debug] Q_st_bar check:
  Shape: torch.Size([6196, 9])
  Row sums min/max: 0.000120 / 0.000200
  Column sums: tensor([0.1287, 0.1224, 0.0981, 0.1498, 0.1214, 0.1073, 0.0814, 0.1032, 0.0876],
       device='cuda:0')
  Min/Max value: 1.409847e-18 / 2.000088e-04
✓ TOS (Eq.5) range: [0.000000, 0.255924]
✓ UOT_conf (w_i) range: [0.000055, 0.000200]
  N_aug = 6196, prior_mass = 1.613944e-04
  Private samples (w_i <= 1.613944e-04): 2305 / 3098 (74.4%)

[Epoch 25] Evaluation:
- Common Acc (OT, High=False): 86.67%
- Overall Acc (OT, High=True): 25.60%


Evaluating on target domain: 100%|██████████| 14/14 [00:00<00:00, 16.76it/s]

Evaluating on source domain: 100%|██████████| 3/3 [00:00<00:00, 24.14it/s]



[Debug] Q_st_bar check:
  Shape: torch.Size([5916, 9])
  Row sums min/max: 0.000105 / 0.000233
  Column sums: tensor([0.1284, 0.1387, 0.0919, 0.1451, 0.1352, 0.0945, 0.0814, 0.1007, 0.0841],
       device='cuda:0')
  Min/Max value: 1.744001e-22 / 2.329201e-04
✓ TOS (Eq.5) range: [0.000000, 0.380220]
✓ UOT_conf (w_i) range: [0.000055, 0.000233]
  N_aug = 5916, prior_mass = 1.690331e-04
  Private samples (w_i <= 1.690331e-04): 1753 / 3098 (56.6%)

[Epoch 50] Evaluation:
- Common Acc (OT, High=False): 89.25%
- Overall Acc (OT, High=True): 42.77%


... (38 outputs left out)


Evaluating on target domain: 100%|██████████| 14/14 [00:00<00:00, 16.53it/s]

Evaluating on source domain: 100%|██████████| 3/3 [00:00<00:00, 23.29it/s]



[Debug] Q_st_bar check:
  Shape: torch.Size([4812, 9])
  Row sums min/max: 0.000082 / 0.000288
  Column sums: tensor([0.2511, 0.1351, 0.0042, 0.2396, 0.2793, 0.0151, 0.0038, 0.0650, 0.0067],
       device='cuda:0')
  Min/Max value: 1.954328e-22 / 2.875257e-04
✓ TOS (Eq.5) range: [0.000000, 0.607453]
✓ UOT_conf (w_i) range: [0.000034, 0.000288]
  N_aug = 4812, prior_mass = 2.078138e-04
  Private samples (w_i <= 2.078138e-04): 638 / 3098 (20.6%)

[Epoch None] Evaluation:
- Common Acc (OT, High=False): 94.77%
- Overall Acc (OT, High=True): 77.37%
✓ ot_matrix shape: (3098, 9)
  Sample row sum: 0.000250
Saving results to ./output/cel_seq2/logs/


global step: 507it [02:28,  3.41it/s]


Loading data from: ./output/cel_seq2/logs/test_results.h5ad

[Class-Balanced Anchor Selection]
  - Class 0: 710 candidates -> 426 Anchors (Thr=0.000225)
  - Class 1: 387 candidates -> 232 Anchors (Thr=0.000269)
  - Class 2: 33 candidates -> 20 Anchors (Thr=0.000123)
  - Class 3: 808 candidates -> 485 Anchors (Thr=0.000232)
  - Class 4: 878 candidates -> 527 Anchors (Thr=0.000251)
  - Class 5: 70 candidates -> 42 Anchors (Thr=0.000150)
  - Class 6: 18 candidates -> 11 Anchors (Thr=0.000101)
  - Class 7: 171 candidates -> 103 Anchors (Thr=0.000172)
  - Class 8: 23 candidates -> 14 Anchors (Thr=0.000153)

[Global Stats] N=3098
 - Anchor Samples:   1860 (balanced across common classes)
 - Boundary Samples: 1238 (require optimization)
 - Private Samples:  0
distill batch size:231
Data N=3098, Batch=231, Remainder=95 => Drop Last: False

Starting training for 200 epochs...
Training finished.

================ Final results comparison ================
1. Teacher (High=True) Acc:  77.37%  (inc

## Inspect measured results
The original PBMC evaluator pools query-private subclasses into one private class before scoring. These are measurements from this run, not a guaranteed manuscript value.

In [4]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
result = sc.read_h5ad(output / 'logs/test_results_distilled.h5ad')
truth = np.minimum(result.obs['labels'].astype(int).to_numpy(), result.obsm['cls_probs'].shape[1])
rows = []
for method, column in [('OT common', 'pred_labels_ot_common'), ('Final distilled', 'final_distill_preds')]:
    pred = result.obs[column].astype(int).to_numpy()
    rows.append({'Method': method, 'Accuracy': accuracy_score(truth, pred), 'Macro_F1': f1_score(truth, pred, average='macro', zero_division=0)})
metrics = pd.DataFrame(rows)
metrics.to_csv(output / 'metrics.csv', index=False)
print(metrics.to_string(index=False))
print(result.obs[['cell_type', 'labels', 'final_distill_preds']].head())


         Method  Accuracy  Macro_F1
      OT common  0.947708  0.901081
Final distilled  0.949645  0.913331
                                         cell_type labels  final_distill_preds
cells                                                                         
pbmc2_10X_V2_AGTGAGGTCTTGAGGT          CD4+ T cell      3                    3
pbmc2_10X_V2_AGTGGGAAGTTCGCAT               B cell      0                    0
pbmc2_10X_V2_AGTGGGACACACCGCA       CD14+ monocyte      1                    1
pbmc2_10X_V2_AGTGTCACACCAACCG               B cell      0                    0
pbmc2_10X_V2_AGTGTCAGTAGAGCTG  Natural killer cell      7                    7
